<a href="https://colab.research.google.com/github/DigoShane/GitHub-ML/blob/main/topologyOpt_AIsoln_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget "https://fem-on-colab.github.io/releases/fenics-install-release-real.sh" -O "/tmp/fenics-install.sh"
!bash "/tmp/fenics-install.sh"

--2026-05-25 05:13:16--  https://fem-on-colab.github.io/releases/fenics-install-release-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4180 (4.1K) [application/x-sh]
Saving to: ‘/tmp/fenics-install.sh’

/tmp/fenics-install 100%[===================>]   4.08K  --.-KB/s    in 0s      

2026-05-25 05:13:16 (43.9 MB/s) - ‘/tmp/fenics-install.sh’ saved [4180/4180]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FENICS_INSTALLED=/usr/local/share/fem-on-colab/fenics.installed
+ [[ ! -f /usr/local/share/fem-on-colab/fenics.installed ]]
+ PYBIND11_INSTALL_SCRIPT_PATH=https://github.com/fem-on-colab/fem-on-colab.github.io/raw/f823fc81/releases/pybi

In [4]:
%matplotlib inline
from dolfin import *
import matplotlib.pyplot as plt
import numpy as np

from google.colab import files

In [5]:
uploaded = files.upload()
data = np.load("solution_data.npz")

nodes = data["nodes"]
elements = data["elements"]
u_data = data["displacements"]

E = float(data["E"])
nu = float(data["nu"])

print("nodes shape        =", nodes.shape)
print("elements shape     =", elements.shape)
print("displacements shape=", u_data.shape)
print("E =", E)
print("nu =", nu)

Saving solution_data.npz to solution_data.npz
nodes shape        = (451, 2)
elements shape     = (800, 3)
displacements shape= (451, 2)
E = 1.0
nu = 0.3


In [6]:
mesh = Mesh()
editor = MeshEditor()

editor.open(mesh, "triangle", 2, 2)

editor.init_vertices(len(nodes))
editor.init_cells(len(elements))

for i, x in enumerate(nodes):
    editor.add_vertex(i, x)

for i, cell in enumerate(elements):
    editor.add_cell(i, cell)

editor.close()

print("Number of vertices:", mesh.num_vertices())
print("Number of cells:", mesh.num_cells())

Number of vertices: 451
Number of cells: 800


In [7]:
V = VectorFunctionSpace(mesh, "CG", 1)

Calling FFC just-in-time (JIT) compiler, this may take some time.


Level 25:FFC:Calling FFC just-in-time (JIT) compiler, this may take some time.
INFO:FFC:Compiling element ffc_element_4f750817ecc896f3bedcb4ff8c9f3352153b1b38

INFO:FFC:Compiler stage 1: Analyzing element(s)
INFO:FFC:--------------------------------------
INFO:FFC:  
INFO:FFC:Compiler stage 1 finished in 0.00284719 seconds.

INFO:FFC:Compiler stage 2: Computing intermediate representation
INFO:FFC:-------------------------------------------------------
INFO:FFC:  Computing representation of 1 elements
DEBUG:FFC:  Reusing element from cache
DEBUG:FFC:  Reusing element from cache
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 1 dofmaps
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 0 coordinate mappings
INFO:FFC:  Computing representation of integrals
INFO:FFC:  Computing representation of forms
INFO:FFC:  
INFO:FFC:Compiler stage 2 finished in 0.48994 seconds.

INFO:FFC:Compiler stage 3: Optimizing intermediate representation

Calling FFC just-in-time (JIT) compiler, this may take some time.


Level 25:FFC:Calling FFC just-in-time (JIT) compiler, this may take some time.
INFO:FFC:Compiling element ffc_element_3801828c0f66b7190a7fd5819465b3d5b34b9149

INFO:FFC:Compiler stage 1: Analyzing element(s)
INFO:FFC:--------------------------------------
INFO:FFC:  
INFO:FFC:Compiler stage 1 finished in 0.0015018 seconds.

INFO:FFC:Compiler stage 2: Computing intermediate representation
INFO:FFC:-------------------------------------------------------
INFO:FFC:  Computing representation of 1 elements
DEBUG:FFC:  Reusing element from cache
DEBUG:FFC:  Reusing element from cache
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 1 dofmaps
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 0 coordinate mappings
INFO:FFC:  Computing representation of integrals
INFO:FFC:  Computing representation of forms
INFO:FFC:  
INFO:FFC:Compiler stage 2 finished in 0.00661016 seconds.

INFO:FFC:Compiler stage 3: Optimizing intermediate representati

In [8]:
u = Function(V)
u_vec = u.vector().get_local()

for i in range(len(nodes)):
    u_vec[2*i]   = u_data[i, 0]
    u_vec[2*i+1] = u_data[i, 1]

u.vector().set_local(u_vec)
u.vector().apply("insert")

In [9]:
plane_stress = False
mu = E/(2.0*(1.0 + nu))

if plane_stress:
    lmbda = E*nu/(1.0 - nu**2)
else:
    lmbda = E*nu/((1.0 + nu)*(1.0 - 2.0*nu))

def eps(v):
    return sym(grad(v))

def sigma(v):
    return 2.0*mu*eps(v) + lmbda*tr(eps(v))*Identity(2)

In [10]:
#Dirichlet BC check at x=-2.
dirichlet_error = 0.0
for i, x in enumerate(nodes):
    if abs(x[0] + 2.0) < 1e-10:
        err = np.linalg.norm(u_data[i] - np.array([0.0, 0.0]))
        dirichlet_error = max(dirichlet_error, err)

print("Max Dirichlet displacement error =", dirichlet_error)

Max Dirichlet displacement error = 0.0
